## **PEFT** (Parameter-Efficient Fine-Tuning)

* Fine-tune LLMs **without touching all parameters**, saves time + compute.
* Works with transformers + LoRA techniques.

**Popular Techniques:**

* **LoRA**: Only update low-rank adapter matrices (adds only a few trainable parameters).
* **QLoRA**: Combines quantization + LoRA → even more memory-efficient.

## 🧠 What is LoRA? (Low-Rank Adaptation)

### 🎯 **Goal**:

Fine-tune a large model (like LLaMA, GPT, etc.) **without updating all of its billions of parameters** — saves time, compute, and memory.

---

### 🚗 Analogy: Customizing a Car Without Rebuilding It

* Imagine you bought a fancy car (like an LLM).
* Instead of replacing the engine to make it faster, you **add a turbo booster on top**.
* The core car stays the same, but performance improves.

🛠️ **LoRA does this by:**

* **Freezing** the original weights (not changing them).
* Adding **small "adapter" layers** (matrices) that **learn the changes**.
* Only training those tiny adapters (\~0.1% of parameters).

---

### 🔢 Technically:

LoRA uses **low-rank matrices** to approximate the changes needed during fine-tuning.

If the original weight matrix is `W`:

* It adds something like: `W + A @ B`
  Where `A` and `B` are small (low-rank) matrices you train.

This is much faster and needs less GPU.

---

## 🧪 What is QLoRA? (Quantized LoRA)

### 🎯 **Goal**:

Make LoRA **even cheaper** by **running it on quantized models** — i.e., compressed models that use less memory (like 4-bit).

---

### 🍫 Analogy: LoRA = tuning your car

QLoRA = **tuning a smaller, lighter version of your car** to save fuel and still get good performance.

---

### 🔍 What’s Special About QLoRA?

* It loads the **base model in 4-bit precision** → very memory-efficient.
* You **still train LoRA adapters** on top.
* Result: You can fine-tune **big models (like LLaMA-7B)** on a **single consumer GPU (like 24GB or even 16GB)**.

---

## ✅ Summary Table

| Feature    | LoRA                             | QLoRA                               |
| ---------- | -------------------------------- | ----------------------------------- |
| Model Size | Full precision (FP16/FP32)       | 4-bit quantized model (smaller)     |
| Training   | Adds adapter layers, freeze base | Same, but on quantized weights      |
| Memory Use | Medium                           | Very low (more efficient)           |
| Use Case   | Fine-tune big models cheaply     | Fine-tune even **cheaper + faster** |

---

## 📦 Tools that support LoRA / QLoRA:

* 🤗 **PEFT (Hugging Face)**
* **AutoGPTQ**, **bitsandbytes** (for quantization)
* **QLoRA paper**: HuggingFace + Tim Dettmers
* Fine-tuning libraries: **TRLLM, Axolotl, LLaMA Factory**

## **First, Quick Recap: Weights**

* Think of weights as **knobs** in a giant control panel (the model).
* In a large LLM, there are **billions** of knobs.
* Normally, to fine-tune a model, you’d adjust **all** those knobs — expensive in memory, compute, and storage.

---

## **LoRA (Low-Rank Adaptation)** – *"Adding a small steering wheel instead of rebuilding the whole bus"*

Instead of touching **all** original weights:

1. Freeze the original big model weights (leave them as they are).
2. Insert **small trainable matrices** (extra knobs) inside specific layers (like attention layers).
3. During fine-tuning, you **only adjust these small matrices**.

**Why?**

* Much smaller memory footprint.
* Training is faster.
* You can store just the small LoRA weights (few MBs instead of GBs) and apply them on top of the base model.

**Analogy:**
Instead of re-cooking the entire recipe every time, you just tweak the **seasoning packet** you sprinkle at the end.

---

## **QLoRA (Quantized LoRA)** – *"Shrinking the bus before adding the steering wheel"*

QLoRA takes LoRA further:

1. **Quantize** the big model’s weights (e.g., from 32-bit to 4-bit) → makes it fit in smaller GPU memory.
2. Still **freeze these quantized weights** (don’t change them during training).
3. Add **LoRA adapters** on top (small trainable knobs).
4. Train only the LoRA parameters — the quantized base just sits there as a frozen reference.

**Why?**

* Lets you fine-tune **huge models on a single GPU** without running out of memory.
* Combines benefits of quantization (memory savings) + LoRA (train small extra weights).

**Analogy:**
Imagine you want to modify a **luxury bus**:

* **Full fine-tune:** rebuild the whole bus → expensive.
* **LoRA:** keep the bus intact, just add a small steering control for special routes.
* **QLoRA:** first make the bus a **lightweight version** (less weight, cheaper to store), then add your steering control.

---

### **Role of Weights in LoRA & QLoRA**

* The **original model weights** remain untouched — they act as the frozen **base knowledge**.
* The **LoRA weights** are tiny add-on adjustments that change the model’s behavior for your specific task.
* In **QLoRA**, the frozen base weights are **quantized** to save memory, but LoRA weights remain full precision (for better fine-tuning quality).

## **Quantization** – "Shrinking the luggage to fit the bus"

Think of an LLM’s **weights** (its learned numbers) like **items in your travel suitcase**.
Normally, each weight is stored as a **32-bit floating-point number** — big, detailed, but **takes up space**.

If your bus (GPU VRAM) is small, you **can’t fit all the suitcases**. So you **compress** the luggage:

* **8-bit quantization** → You keep the most important details but store them more compactly.
* **4-bit quantization** → Even smaller, at the cost of a little accuracy.

The result:

* **Fits into smaller memory** (can run big models on smaller GPUs or even CPU).
* **Runs faster** (less data to move around).

**Example:**

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,                 # store weights in 4-bit format
    bnb_4bit_use_double_quant=True,    # extra compression layer
    bnb_4bit_compute_dtype=torch.bfloat16, # compute in bfloat16 for speed
    bnb_4bit_quant_type="nf4"          # special 4-bit format
)
model = AutoModelForCausalLM.from_pretrained("model_name", quantization_config=quant_config)


**Analogy:**

* Original: 32-bit = **high-res photo**.
* Quantized: 4-bit = **pixel art version** — still recognizable, but lighter to carry.

* **Quantization** → shrink suitcases so the bus can carry them.

I'll explain LoRA (Low-Rank Adaptation) and QLoRA (Quantized LoRA) step by step with Python examples.

## What is LoRA?

LoRA is a parameter-efficient fine-tuning technique that reduces the number of trainable parameters when adapting large language models. Instead of updating all model weights, LoRA learns low-rank decomposition matrices that are added to the original weights.

### Key Concept
Instead of updating a weight matrix W directly, LoRA decomposes the update into two smaller matrices:
- **ΔW = BA** where B is (d × r) and A is (r × k), with r << min(d,k)
- The new forward pass becomes: **h = W₀x + ΔWx = W₀x + BAx**

## What is QLoRA?

QLoRA (Quantized LoRA) combines LoRA with 4-bit quantization to further reduce memory usage. It uses:
1. **4-bit NormalFloat (NF4)** quantization for base model weights
2. **Double quantization** for quantization constants
3. **Paged optimizers** for memory management## Step-by-Step Comparison

### LoRA Process:
1. **Freeze base model weights** - Original parameters aren't updated
2. **Add low-rank matrices** - Learn A (rank × input_dim) and B (output_dim × rank) 
3. **Forward pass** - output = W₀x + BAx
4. **Train only LoRA parameters** - Reduces trainable params by ~99%

### QLoRA Process:
1. **Quantize base model to 4-bit** - Use NF4 quantization + double quantization
2. **Keep LoRA adapters in 16-bit** - Maintain training precision
3. **Use paged optimizers** - Manage GPU memory efficiently
4. **Gradient computation** - Dequantize → compute → quantize## Key Benefits and Use Cases

### LoRA Advantages:
- **Parameter Efficiency**: Reduces trainable parameters by 99%+
- **Memory Savings**: Lower memory requirements during training
- **Modular**: Can save/load different LoRA adapters for different tasks
- **Quality**: Maintains performance comparable to full fine-tuning

### QLoRA Advantages:
- **Extreme Memory Efficiency**: 4x additional memory reduction
- **Accessibility**: Enables fine-tuning of 65B+ models on single GPUs
- **Cost Effective**: Significantly reduces hardware requirements
- **Practical**: Makes large model fine-tuning accessible to more researchers

### When to Use Which:
- **Full Fine-tuning**: When you have abundant resources and need maximum performance
- **LoRA**: When you need good performance with moderate resource constraints
- **QLoRA**: When you have limited GPU memory but still want to fine-tune large models

I'll explain LoRA and QLoRA separately, one by one, with detailed explanations and code examples.

# LoRA (Low-Rank Adaptation)

## What is LoRA?

LoRA is a parameter-efficient fine-tuning technique that allows you to adapt large pre-trained models without updating all their parameters. Instead of modifying the original weights directly, LoRA learns small "adapter" matrices that are added to the original weights.

## Core Mathematical Concept

The key insight is that weight updates during fine-tuning often have a low "intrinsic rank" - meaning they can be represented efficiently using matrix decomposition.

**Traditional Fine-tuning:**
- Update: W = W₀ + ΔW (where ΔW is full-rank)
- All parameters need gradients and storage

**LoRA Approach:**
- Update: W = W₀ + ΔW = W₀ + B·A
- Where A is (r × d) and B is (d × r), with r << d
- Only A and B need gradients (much smaller!)

## Step-by-Step LoRA Process

### Step 1: Matrix Decomposition
- Original update: **ΔW** (full rank matrix)
- LoRA update: **ΔW = B·A** where **A** ∈ ℝ^(r×d), **B** ∈ ℝ^(d×r)
- Rank **r** << **d** (typically r=4-64, d=1024-4096)

### Step 2: Parameter Freezing
- Freeze all original model weights: `param.requires_grad = False`
- Only train the small LoRA matrices A and B
- Reduces trainable parameters by 99%+

### Step 3: Scaling Factor
- Apply scaling: **α/r** where α is typically 16-32
- Controls the magnitude of LoRA adaptations
- Higher α = stronger adaptation, lower α = more conservative

### Step 4: Forward Pass
- Compute: **h = W₀x + α/r · B(A(x))**
- More efficient to compute **A(x)** first, then **B(·)**
- Avoids materializing the full **B·A** matrix

---

# QLoRA (Quantized LoRA)

## What is QLoRA?

QLoRA extends LoRA by adding 4-bit quantization to the base model, making it possible to fine-tune very large models (65B+ parameters) on consumer hardware. It combines three key innovations:

1. **4-bit NormalFloat (NF4) quantization** for base model weights
2. **Double quantization** for quantization constants  
3. **Paged optimizers** for memory management

## Step-by-Step QLoRA Process

### Step 1: 4-bit Quantization
- Convert base model weights from FP16 (2 bytes) to NF4 (0.5 bytes)
- **NF4 (NormalFloat4)**: 16 quantization levels optimized for normal distributions
- Store quantization indices (4-bit) + scale factors (FP16)

### Step 2: Double Quantization
- Further quantize the scale factors themselves to 8-bit
- Reduces memory overhead of quantization constants
- Saves additional ~0.4 bits per parameter

### Step 3: Mixed Precision Training
- **Base model**: 4-bit quantized (frozen)
- **LoRA adapters**: 16-bit precision (trainable)
- **Gradients**: Computed in 16-bit, then applied to adapters

### Step 4: Paged Optimizers
- Store optimizer states in CPU memory when not needed
- Page them to GPU only during gradient updates
- Prevents memory fragmentation and enables larger models

## Key Differences Summary

| Aspect | LoRA | QLoRA |
|--------|------|--------|
| **Base Model** | FP16 (frozen) | 4-bit NF4 (frozen) |
| **Adapters** | FP16 (trainable) | FP16 (trainable) |
| **Memory Usage** | ~50% of full fine-tuning | ~25% of LoRA |
| **Training Speed** | Fast | Slightly slower (quantization overhead) |
| **Quality** | ~100% of full fine-tuning | ~95-99% of full fine-tuning |
| **Hardware Requirements** | 8-16 GB GPU | 4-8 GB GPU |

## When to Use Each

**Use LoRA when:**
- You have sufficient GPU memory (8+ GB)
- You need maximum training speed
- You're working with smaller models (<7B parameters)
- You want to minimize any potential quality loss

**Use QLoRA when:**
- You have limited GPU memory (4-8 GB)
- You want to fine-tune very large models (13B+ parameters)
- Memory efficiency is more important than speed
- You can accept minimal quality trade-offs

I'll create a simple, step-by-step explanation of LoRA with easy-to-understand code examples.This simple explanation covers LoRA step by step with easy-to-understand code:

## 🎯 **The Core Idea**
Instead of updating ALL weights in a neural network (expensive), LoRA updates only two small matrices A and B that multiply together to approximate the full update.

## 📐 **The Math (Simple Version)**
- **Traditional**: `W_new = W_old + ΔW` (update entire matrix)
- **LoRA**: `W_new = W_old + B × A` (update via two small matrices)
- **Savings**: From N×N parameters down to 2×rank×N parameters

## 💡 **Visual Example**
If you have a 1000×1000 weight matrix:
- **Traditional**: Need 1,000,000 parameters to update
- **LoRA (rank=16)**: Need only 32,000 parameters (31x reduction!)

## 🔑 **Key Insights**
1. **Most changes are "low-rank"** - you don't need to change every single weight
2. **Pre-trained models are smart** - they already know most of what they need
3. **Small adjustments work** - you just need to nudge them in the right direction

## 🎓 **Why It's Brilliant**
- **Memory**: Uses 99% less trainable parameters
- **Speed**: Trains much faster
- **Flexibility**: Can save different LoRA adapters for different tasks
- **Quality**: Performs almost as well as full fine-tuning

The code shows this progression from basic matrix math to practical implementation, making it clear how this "simple" idea revolutionized AI fine-tuning!

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("🎯 LoRA (Low-Rank Adaptation) - Simple Explanation")
print("=" * 60)

# ============================================================================
# STEP 1: The Problem - Why do we need LoRA?
# ============================================================================

print("\n📚 STEP 1: The Problem")
print("-" * 30)

# Imagine we have a pre-trained model with a large weight matrix
original_size = 1000  # This could be 4096 in real models like GPT
weight_matrix = torch.randn(original_size, original_size) * 0.02

print(f"Original weight matrix size: {weight_matrix.shape}")
print(f"Number of parameters: {weight_matrix.numel():,}")
print(f"Memory needed (32-bit): {weight_matrix.numel() * 4 / 1e6:.1f} MB")

print("\n❌ Traditional Fine-tuning Problem:")
print("• Need to update ALL parameters")
print("• Requires storing gradients for ALL parameters")
print("• Uses LOTS of memory")
print("• Need to save entire model for each task")

# ============================================================================
# STEP 2: The LoRA Idea - Low-Rank Decomposition
# ============================================================================

print("\n\n💡 STEP 2: The LoRA Solution")
print("-" * 30)

print("🔑 Key Insight: Most weight updates are 'low-rank'")
print("Instead of updating the full matrix W, we can approximate the update as:")
print("    W_new = W_original + ΔW")
print("    where ΔW = A × B")
print("    A is small matrix (rank × original_size)")  
print("    B is small matrix (original_size × rank)")

# Let's see this with numbers
rank = 16  # This is much smaller than original_size (1000)

print(f"\nWith rank = {rank}:")
print(f"A matrix size: ({rank} × {original_size}) = {rank * original_size:,} parameters")
print(f"B matrix size: ({original_size} × {rank}) = {original_size * rank:,} parameters")
print(f"Total LoRA parameters: {2 * rank * original_size:,}")

original_params = original_size * original_size
lora_params = 2 * rank * original_size
reduction = original_params / lora_params

print(f"\n📊 Comparison:")
print(f"Original parameters: {original_params:,}")
print(f"LoRA parameters: {lora_params:,}")
print(f"Reduction factor: {reduction:.1f}x")
print(f"Memory savings: {(1 - 1/reduction)*100:.1f}%")

# ============================================================================
# STEP 3: Simple LoRA Implementation
# ============================================================================

print("\n\n🔧 STEP 3: Simple LoRA Implementation")
print("-" * 30)

class SimpleLoRA(nn.Module):
    """
    The simplest possible LoRA implementation
    """
    def __init__(self, original_size, rank=16):
        super().__init__()
        
        # Original weight matrix (this would be frozen in practice)
        self.original_weight = nn.Parameter(torch.randn(original_size, original_size) * 0.02)
        
        # LoRA matrices - these are the ONLY things we train!
        self.A = nn.Parameter(torch.randn(rank, original_size) * 0.01)  # Small random values
        self.B = nn.Parameter(torch.zeros(original_size, rank))         # Start with zeros
        
        self.rank = rank
        
        # In real LoRA, we freeze the original weights
        self.original_weight.requires_grad = False  # Don't train original weights!
    
    def forward(self, x):
        # Original computation: x @ W_original
        original_output = x @ self.original_weight
        
        # LoRA computation: x @ (A.T @ B.T) = x @ B.T @ A.T
        # We do this step by step to be memory efficient:
        # Step 1: x @ B.T (x goes through B first)
        # Step 2: result @ A.T (then through A)
        lora_output = x @ self.B @ self.A
        
        # Final output combines both
        return original_output + lora_output

# Let's test it!
print("Testing SimpleLoRA:")
model = SimpleLoRA(original_size=100, rank=8)  # Smaller example

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")
print(f"Trainable percentage: {trainable_params/total_params*100:.1f}%")

# Test forward pass
batch_size = 5
input_features = 100
x = torch.randn(batch_size, input_features)
output = model(x)
print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")

# ============================================================================
# STEP 4: Understanding the Mathematics
# ============================================================================

print("\n\n📐 STEP 4: Understanding the Math")
print("-" * 30)

def demonstrate_matrix_decomposition():
    """Show how LoRA approximates a full matrix update"""
    
    print("Let's see how LoRA approximates a weight update:")
    
    # Original matrix (small example)
    W_original = torch.randn(4, 4)
    
    # Suppose we want to update it by some amount
    full_update = torch.randn(4, 4) * 0.1
    W_full_update = W_original + full_update
    
    # LoRA approximation with rank 2
    rank = 2
    A = torch.randn(rank, 4) * 0.1
    B = torch.randn(4, rank) * 0.1
    lora_update = B @ A  # This is our approximation of full_update
    W_lora = W_original + lora_update
    
    print(f"Original matrix:\n{W_original.numpy()}")
    print(f"\nFull update matrix:\n{full_update.numpy()}")
    print(f"\nLoRA approximation:\n{lora_update.numpy()}")
    
    # Calculate how good the approximation is
    error = torch.mean((full_update - lora_update)**2)
    print(f"\nApproximation error (MSE): {error:.6f}")
    
    # Parameter comparison
    full_params = full_update.numel()
    lora_params = A.numel() + B.numel()
    print(f"\nParameters needed:")
    print(f"Full update: {full_params}")
    print(f"LoRA (A + B): {lora_params}")
    print(f"Reduction: {full_params / lora_params:.1f}x")

demonstrate_matrix_decomposition()

# ============================================================================
# STEP 5: Practical LoRA Layer
# ============================================================================

print("\n\n🛠️ STEP 5: Practical LoRA Layer")
print("-" * 30)

class PracticalLoRA(nn.Module):
    """
    A more practical LoRA implementation like what you'd actually use
    """
    def __init__(self, original_layer, rank=16, alpha=32):
        super().__init__()
        
        # Keep reference to original layer
        self.original_layer = original_layer
        
        # Freeze original layer
        for param in original_layer.parameters():
            param.requires_grad = False
        
        # Get dimensions
        in_features = original_layer.in_features
        out_features = original_layer.out_features
        
        # LoRA parameters
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank  # This controls how strong the adaptation is
        
        # LoRA matrices implemented as Linear layers (easier to work with)
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        
        # Initialize weights properly
        nn.init.kaiming_uniform_(self.lora_A.weight)  # Random small values
        nn.init.zeros_(self.lora_B.weight)           # Start with zeros
    
    def forward(self, x):
        # Original layer output (frozen)
        original_out = self.original_layer(x)
        
        # LoRA pathway: x -> A -> B -> scale
        lora_out = self.lora_B(self.lora_A(x)) * self.scaling
        
        return original_out + lora_out

# Example usage
print("Creating practical LoRA example:")
original_linear = nn.Linear(512, 512)
lora_layer = PracticalLoRA(original_linear, rank=16, alpha=32)

print(f"Original layer parameters: {sum(p.numel() for p in original_linear.parameters()):,}")
print(f"LoRA trainable parameters: {sum(p.numel() for p in lora_layer.parameters() if p.requires_grad):,}")

# Test it
x = torch.randn(10, 512)
output = lora_layer(x)
print(f"Input: {x.shape} -> Output: {output.shape}")

# ============================================================================
# STEP 6: Why Does This Work?
# ============================================================================

print("\n\n🤔 STEP 6: Why Does LoRA Work So Well?")
print("-" * 30)

print("🔬 Scientific Intuition:")
print("1. Most fine-tuning updates are 'low-rank'")
print("   - The model only needs to learn a few new 'directions'")
print("   - Not every parameter needs to change significantly")

print("\n2. Weight matrices have intrinsic low dimensionality")
print("   - Large matrices often have redundant information")
print("   - The important changes can be captured with fewer parameters")

print("\n3. Task adaptation requires limited changes")
print("   - Pre-trained models already know a lot")
print("   - We only need small adjustments for new tasks")

def demonstrate_rank_effects():
    """Show how different ranks affect capacity"""
    print("\n📊 Rank Effect Demonstration:")
    
    # Create a target update we want to learn
    size = 100
    target_update = torch.randn(size, size) * 0.1
    
    ranks_to_test = [1, 4, 16, 32, 64]
    
    print(f"{'Rank':<6} {'Parameters':<12} {'Approximation Error':<20}")
    print("-" * 40)
    
    for rank in ranks_to_test:
        # Create LoRA matrices
        A = torch.randn(rank, size) * 0.1
        B = torch.randn(size, rank) * 0.1
        
        # LoRA approximation
        lora_approx = B @ A
        
        # Calculate error
        error = torch.mean((target_update - lora_approx)**2).item()
        params = rank * size * 2
        
        print(f"{rank:<6} {params:<12,} {error:<20.6f}")
    
    print("\n💡 Notice: Higher rank = better approximation but more parameters!")

demonstrate_rank_effects()

# ============================================================================
# STEP 7: Simple Training Example
# ============================================================================

print("\n\n🎓 STEP 7: Simple Training Example")
print("-" * 30)

def simple_training_demo():
    """Show how to train a LoRA model"""
    
    # Create a simple task: learn to double the input
    def target_function(x):
        return x * 2
    
    # Create model
    original_layer = nn.Linear(10, 10)
    lora_model = PracticalLoRA(original_layer, rank=4, alpha=8)
    
    # Training setup
    optimizer = torch.optim.Adam(lora_model.parameters(), lr=0.01)
    criterion = nn.MSELoss()
    
    print("Training LoRA to double inputs...")
    print(f"Trainable parameters: {sum(p.numel() for p in lora_model.parameters() if p.requires_grad):,}")
    
    # Training loop
    for epoch in range(100):
        # Generate random data
        x = torch.randn(32, 10)
        target = target_function(x)
        
        # Forward pass
        prediction = lora_model(x)
        loss = criterion(prediction, target)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch % 20 == 0:
            print(f"Epoch {epoch}: Loss = {loss.item():.6f}")
    
    print("Training completed!")
    
    # Test the model
    test_input = torch.tensor([[1., 2., 3., 4., 5., 6., 7., 8., 9., 10.]])
    with torch.no_grad():
        result = lora_model(test_input)
    
    print(f"\nTest:")
    print(f"Input:  {test_input.numpy()}")
    print(f"Output: {result.numpy()}")
    print(f"Target: {(test_input * 2).numpy()}")

simple_training_demo()

# ============================================================================
# SUMMARY
# ============================================================================

print("\n\n🎯 SUMMARY: What is LoRA?")
print("=" * 50)

print("✅ LoRA breaks down weight updates into two small matrices:")
print("   W_new = W_original + B × A")

print("\n✅ This gives us HUGE parameter reduction:")
print("   • Original: N × N parameters")
print("   • LoRA: 2 × rank × N parameters") 
print("   • Typical reduction: 100x - 1000x fewer parameters!")

print("\n✅ Why it works:")
print("   • Most weight updates are low-rank")
print("   • Fine-tuning needs only small adjustments")
print("   • Pre-trained models already know most of what they need")

print("\n✅ Benefits:")
print("   • Much less memory needed")
print("   • Faster training")
print("   • Can save different adapters for different tasks")
print("   • Almost same performance as full fine-tuning")

print("\n🚀 That's LoRA in a nutshell!")
print("   Simple idea, powerful results! 🎉")


I'll create a simple, step-by-step explanation of QLoRA that builds on the LoRA concepts.This simple QLoRA explanation builds on the LoRA concepts and shows:

## 🎯 **The Core QLoRA Innovation**
QLoRA = LoRA + 4-bit quantization of the base model, giving you the best of both worlds:
- **LoRA**: 99% fewer trainable parameters  
- **4-bit quantization**: 8x less memory for base model

## 🔢 **4-bit Quantization Made Simple**
Instead of storing each weight as a 32-bit number, store it as one of 16 predefined values (4 bits):
- **Regular**: `0.1234` (32 bits)
- **4-bit**: Choose closest from 16 values (4 bits)
- **Memory**: 8x reduction!

## 🧠 **NF4: Smart Quantization**
- Regular quantization: evenly spaced values
- **NF4**: More values near zero (where most weights are)
- **Result**: Much better approximation for neural networks

## 💾 **Memory Comparison (7B Model)**

Full Fine-tuning: 28 GB
LoRA:            14 GB  
QLoRA:           3.5 GB  ← Fits on RTX 3070!


## 🎓 **The Training Process**
1. **Base model**: Quantized to 4-bit, frozen
2. **LoRA adapters**: Stay in 16-bit, trainable
3. **Forward pass**: Dequantize → compute → add LoRA
4. **Backward pass**: Only LoRA gets gradients

## ✨ **Why QLoRA Works**
- Neural networks are **robust** to quantization noise
- LoRA adapters **compensate** for any quality loss
- **4-bit is the sweet spot**: 2-bit too lossy, 8-bit unnecessary

The code progression shows how simple quantization evolves into sophisticated NF4 quantization, then combines with LoRA to create the QLoRA breakthrough that democratized large model fine-tuning!

In [ ]:
import torch
import torch.nn as nn
import numpy as np

print("🎯 QLoRA (Quantized Low-Rank Adaptation) - Simple Explanation")
print("=" * 65)

# ============================================================================
# STEP 1: Quick LoRA Recap + The New Problem
# ============================================================================

print("\n📚 STEP 1: From LoRA to QLoRA - The Problem")
print("-" * 45)

print("🔄 Quick LoRA Recap:")
print("• LoRA reduced trainable parameters by 99%")
print("• But the BASE MODEL still takes lots of memory")
print("• Example: 7B parameter model = 14GB in 16-bit")

# Let's see the memory breakdown
model_size_7B = 7_000_000_000  # 7 billion parameters
memory_16bit = model_size_7B * 2 / 1e9  # 2 bytes per parameter
memory_32bit = model_size_7B * 4 / 1e9  # 4 bytes per parameter

print(f"\n📊 Memory Usage for 7B Model:")
print(f"• 32-bit (FP32): {memory_32bit:.1f} GB")
print(f"• 16-bit (FP16): {memory_16bit:.1f} GB")
print(f"• Still too much for most GPUs! 😵")

print(f"\n❌ The Problem:")
print("• Even with LoRA, base model uses too much memory")
print("• Most people can't afford 24GB+ GPUs")
print("• Can't fine-tune large models on consumer hardware")

print(f"\n💡 QLoRA Solution:")
print("• Keep LoRA (small adapters)")
print("• + Quantize base model to 4-bit!")
print("• 4-bit = 0.5 bytes per parameter")

memory_4bit = model_size_7B * 0.5 / 1e9
print(f"• 4-bit memory: {memory_4bit:.1f} GB ✨")
print(f"• Reduction: {memory_16bit/memory_4bit:.0f}x less memory!")

# ============================================================================
# STEP 2: Understanding 4-bit Quantization
# ============================================================================

print("\n\n🔢 STEP 2: What is 4-bit Quantization?")
print("-" * 45)

print("🎨 Think of it like reducing colors in an image:")
print("• Original: millions of colors (32-bit numbers)")
print("• Quantized: only 16 colors (4-bit numbers)")
print("• Image still recognizable, but much smaller!")

def demonstrate_simple_quantization():
    """Show basic quantization concept"""
    print("\n🔍 Simple Quantization Demo:")
    
    # Original weights (full precision)
    original_weights = torch.tensor([0.1234, -0.0987, 0.0456, -0.1111, 0.0789])
    print(f"Original weights: {original_weights}")
    print(f"Memory per weight: 32 bits (4 bytes)")
    
    # Simple 4-bit quantization (16 levels: -8 to 7)
    # Step 1: Find the range
    max_val = original_weights.abs().max()
    
    # Step 2: Scale to 4-bit range (-8 to 7)
    scale_factor = max_val / 7
    scaled = original_weights / scale_factor
    
    # Step 3: Round to integers
    quantized_ints = torch.round(scaled).clamp(-8, 7).int()
    
    # Step 4: Convert back to approximate original values
    quantized_weights = quantized_ints.float() * scale_factor
    
    print(f"Quantized ints:   {quantized_ints} (4 bits each)")
    print(f"Quantized weights: {quantized_weights}")
    print(f"Scale factor:     {scale_factor:.4f}")
    print(f"Memory per weight: 4 bits (0.5 bytes)")
    print(f"Memory reduction:  8x!")
    
    # Show the error
    error = torch.abs(original_weights - quantized_weights)
    print(f"Quantization error: {error}")
    print(f"Max error: {error.max():.4f}")

demonstrate_simple_quantization()

# ============================================================================
# STEP 3: NF4 - Smart 4-bit Quantization
# ============================================================================

print("\n\n🧠 STEP 3: NF4 - Smarter 4-bit Quantization")
print("-" * 45)

print("🤖 Problem with simple quantization:")
print("• Neural network weights aren't evenly distributed")
print("• Most weights are close to zero (normal distribution)")
print("• Simple quantization wastes 'slots' on rare large values")

print("\n✨ NF4 (NormalFloat4) Solution:")
print("• Use 16 levels optimized for normal distribution")
print("• More levels near zero, fewer for extreme values")
print("• Much better approximation for neural networks!")

class SimpleNF4:
    """Simplified NF4 quantization for demonstration"""
    
    # These 16 values are optimized for normal distributions
    NF4_VALUES = torch.tensor([
        -1.0, -0.6962, -0.5251, -0.3949,  # Negative values
        -0.2844, -0.1848, -0.0911, 0.0,   # Near zero
        0.0796, 0.1609, 0.2461, 0.3379,   # Small positive  
        0.4407, 0.5626, 0.7230, 1.0       # Larger positive
    ])
    
    @classmethod
    def quantize(cls, weights):
        """Quantize weights to NF4 format"""
        # Step 1: Find scale (max absolute value)
        scale = weights.abs().max()
        if scale == 0:
            scale = 1.0
        
        # Step 2: Normalize to [-1, 1]
        normalized = weights / scale
        
        # Step 3: Find closest NF4 value for each weight
        distances = torch.abs(normalized.unsqueeze(-1) - cls.NF4_VALUES.unsqueeze(0))
        indices = torch.argmin(distances, dim=-1)
        
        # Step 4: Get quantized values
        quantized = cls.NF4_VALUES[indices] * scale
        
        return quantized, indices, scale
    
    @classmethod
    def dequantize(cls, indices, scale):
        """Convert back from indices to approximate weights"""
        return cls.NF4_VALUES[indices] * scale

def demonstrate_nf4():
    """Show NF4 vs simple quantization"""
    print("\n🔍 NF4 vs Simple Quantization Demo:")
    
    # Create neural network-like weights (normal distribution)
    torch.manual_seed(42)
    nn_weights = torch.randn(20) * 0.1  # Typical NN weight scale
    
    print(f"Original weights: {nn_weights[:5]}...")  # Show first 5
    print(f"Mean: {nn_weights.mean():.4f}, Std: {nn_weights.std():.4f}")
    
    # NF4 quantization
    nf4_quantized, nf4_indices, nf4_scale = SimpleNF4.quantize(nn_weights)
    
    # Simple uniform quantization for comparison
    max_val = nn_weights.abs().max()
    simple_quantized = torch.round(nn_weights / max_val * 7).clamp(-8, 7)
    simple_quantized = simple_quantized / 7 * max_val
    
    # Compare errors
    nf4_error = torch.abs(nn_weights - nf4_quantized).mean()
    simple_error = torch.abs(nn_weights - simple_quantized).mean()
    
    print(f"\nQuantization Results:")
    print(f"NF4 error:    {nf4_error:.6f}")
    print(f"Simple error: {simple_error:.6f}")
    print(f"NF4 is {simple_error/nf4_error:.1f}x better! 🎯")
    
    return nf4_quantized, nf4_indices, nf4_scale

nf4_result = demonstrate_nf4()

# ============================================================================
# STEP 4: Building a Quantized Layer
# ============================================================================

print("\n\n🏗️ STEP 4: Building a Quantized Linear Layer")
print("-" * 45)

class QuantizedLinear(nn.Module):
    """A Linear layer that stores weights in 4-bit format"""
    
    def __init__(self, in_features, out_features):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Storage for quantized weights
        self.register_buffer('weight_indices', torch.zeros((out_features, in_features), dtype=torch.int8))
        self.register_buffer('weight_scales', torch.zeros(out_features))
        
        # We'll initialize with random weights and then quantize them
        self._initialize_weights()
    
    def _initialize_weights(self):
        """Initialize and quantize the weights"""
        # Create random weights like a normal Linear layer
        temp_weight = torch.randn(self.out_features, self.in_features) * 0.02
        
        # Quantize each row separately (better compression)
        for i in range(self.out_features):
            row = temp_weight[i]
            _, indices, scale = SimpleNF4.quantize(row)
            self.weight_indices[i] = indices
            self.weight_scales[i] = scale
    
    def get_dequantized_weight(self):
        """Get the full-precision weights for computation"""
        weight = torch.zeros_like(self.weight_indices, dtype=torch.float32)
        
        for i in range(self.out_features):
            weight[i] = SimpleNF4.dequantize(
                self.weight_indices[i], 
                self.weight_scales[i]
            )
        
        return weight
    
    def forward(self, x):
        # Dequantize weights for computation
        weight = self.get_dequantized_weight()
        return torch.nn.functional.linear(x, weight)

# Test the quantized layer
print("Testing QuantizedLinear:")
quantized_layer = QuantizedLinear(100, 50)

# Compare memory usage
normal_layer = nn.Linear(100, 50)
normal_memory = sum(p.numel() * 4 for p in normal_layer.parameters())  # 4 bytes per param
quantized_memory = (quantized_layer.weight_indices.numel() * 0.5 +  # 4-bit indices  
                   quantized_layer.weight_scales.numel() * 4)        # 32-bit scales

print(f"Normal layer memory:    {normal_memory:,} bytes")
print(f"Quantized layer memory: {quantized_memory:,} bytes") 
print(f"Memory reduction:       {normal_memory/quantized_memory:.1f}x")

# Test that it works
x = torch.randn(10, 100)
output = quantized_layer(x)
print(f"Input: {x.shape} -> Output: {output.shape} ✅")

# ============================================================================
# STEP 5: The Complete QLoRA Layer
# ============================================================================

print("\n\n🎯 STEP 5: Complete QLoRA = Quantized Base + LoRA Adapters")
print("-" * 45)

class QLoRALinear(nn.Module):
    """
    QLoRA: Quantized base layer + LoRA adapters
    """
    
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        
        # Quantized base layer (frozen, 4-bit)
        self.base_layer = QuantizedLinear(in_features, out_features)
        
        # Freeze the base layer
        for param in self.base_layer.parameters():
            param.requires_grad = False
        
        # LoRA adapters (trainable, 16-bit)
        self.rank = rank
        self.alpha = alpha
        self.scaling = alpha / rank
        
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        
        # Initialize LoRA weights
        nn.init.kaiming_uniform_(self.lora_A.weight)
        nn.init.zeros_(self.lora_B.weight)
    
    def forward(self, x):
        # Base output (from 4-bit quantized weights)
        base_output = self.base_layer(x)
        
        # LoRA output (from 16-bit adapter weights)
        lora_output = self.lora_B(self.lora_A(x)) * self.scaling
        
        return base_output + lora_output

# Test QLoRA
print("Testing QLoRALinear:")
qlora_layer = QLoRALinear(512, 512, rank=16, alpha=32)

# Parameter analysis
total_params = sum(p.numel() for p in qlora_layer.parameters())
trainable_params = sum(p.numel() for p in qlora_layer.parameters() if p.requires_grad)
frozen_params = total_params - trainable_params

print(f"Total parameters:     {total_params:,}")
print(f"Trainable (LoRA):     {trainable_params:,}")
print(f"Frozen (quantized):   {frozen_params:,}")
print(f"Trainable percentage: {trainable_params/total_params*100:.2f}%")

# Memory analysis
base_memory = (qlora_layer.base_layer.weight_indices.numel() * 0.5 + 
               qlora_layer.base_layer.weight_scales.numel() * 4)
lora_memory = sum(p.numel() * 2 for p in [qlora_layer.lora_A.weight, qlora_layer.lora_B.weight])
total_memory = base_memory + lora_memory

print(f"\nMemory breakdown:")
print(f"Base layer (4-bit):   {base_memory:,.0f} bytes")
print(f"LoRA adapters (16-bit): {lora_memory:,.0f} bytes")
print(f"Total QLoRA memory:   {total_memory:,.0f} bytes")

# Compare with regular Linear layer
regular_memory = 512 * 512 * 4  # 32-bit weights
print(f"Regular layer (32-bit): {regular_memory:,} bytes")
print(f"QLoRA memory reduction: {regular_memory/total_memory:.1f}x! 🚀")

# ============================================================================
# STEP 6: Double Quantization - Going Even Further
# ============================================================================

print("\n\n🔬 STEP 6: Double Quantization - Quantizing the Quantizers!")
print("-" * 45)

print("🤔 Wait, there's more we can save!")
print("• We store scale factors in 32-bit (4 bytes each)")
print("• For big models, that's still a lot of scales!")
print("• Solution: Quantize the scales too! (8-bit)")

class DoubleQuantizedNF4:
    """NF4 with double quantization - quantize the scale factors too!"""
    
    @classmethod
    def quantize_scales(cls, scales):
        """Quantize the scale factors to 8-bit"""
        if len(scales) == 0:
            return scales, 0, 1
            
        scale_min = scales.min()
        scale_max = scales.max()
        
        if scale_max == scale_min:
            return torch.zeros_like(scales, dtype=torch.uint8), scale_min, scale_max
        
        # Quantize to 8-bit (0-255)
        normalized = (scales - scale_min) / (scale_max - scale_min)
        quantized = (normalized * 255).round().clamp(0, 255).to(torch.uint8)
        
        return quantized, scale_min, scale_max
    
    @classmethod
    def dequantize_scales(cls, quantized_scales, scale_min, scale_max):
        """Dequantize scale factors back to float"""
        if scale_max == scale_min:
            return torch.full_like(quantized_scales, scale_min, dtype=torch.float32)
        
        normalized = quantized_scales.float() / 255.0
        return normalized * (scale_max - scale_min) + scale_min

def demonstrate_double_quantization():
    """Show double quantization benefits"""
    print("\n🔍 Double Quantization Demo:")
    
    # Simulate scale factors from a model
    num_layers = 100
    scales = torch.rand(num_layers) * 0.1 + 0.01  # Typical scale range
    
    print(f"Original scales: {scales[:5]}...")
    print(f"Original memory: {scales.numel() * 4} bytes (32-bit)")
    
    # Apply double quantization
    q_scales, scale_min, scale_max = DoubleQuantizedNF4.quantize_scales(scales)
    reconstructed = DoubleQuantizedNF4.dequantize_scales(q_scales, scale_min, scale_max)
    
    # Memory comparison
    original_memory = scales.numel() * 4  # 32-bit
    quantized_memory = (q_scales.numel() * 1 +  # 8-bit quantized scales
                       2 * 4)                    # min/max values
    
    print(f"Quantized memory: {quantized_memory} bytes (8-bit + 2 floats)")
    print(f"Memory reduction: {original_memory/quantized_memory:.1f}x")
    
    # Accuracy check
    error = torch.abs(scales - reconstructed).mean()
    print(f"Reconstruction error: {error:.6f}")

demonstrate_double_quantization()

# ============================================================================
# STEP 7: Memory Comparison - The Big Picture
# ============================================================================

print("\n\n📊 STEP 7: The Big Picture - Memory Comparison")
print("-" * 45)

def memory_comparison_7b_model():
    """Compare memory usage for different approaches on a 7B model"""
    
    # Model specs (approximate for 7B model)
    num_params = 7_000_000_000
    num_layers = 32
    d_model = 4096
    
    print(f"Analyzing 7B parameter model:")
    print(f"• {num_params:,} total parameters")
    print(f"• {num_layers} layers")
    print(f"• {d_model} model dimension")
    
    approaches = {
        "Full Fine-tuning (32-bit)": {
            "base_bits": 32,
            "adapter_bits": 0,
            "trainable_ratio": 1.0,
            "rank": None
        },
        "Full Fine-tuning (16-bit)": {
            "base_bits": 16, 
            "adapter_bits": 0,
            "trainable_ratio": 1.0,
            "rank": None
        },
        "LoRA (16-bit base)": {
            "base_bits": 16,
            "adapter_bits": 16,
            "trainable_ratio": 0.01,  # ~1% trainable
            "rank": 16
        },
        "QLoRA (4-bit base)": {
            "base_bits": 4,
            "adapter_bits": 16, 
            "trainable_ratio": 0.01,
            "rank": 16
        }
    }
    
    print(f"\n{'Method':<25} {'Memory (GB)':<12} {'Trainable %':<12} {'GPU Needed':<15}")
    print("-" * 70)
    
    for method, config in approaches.items():
        # Base model memory
        base_memory_gb = num_params * config["base_bits"] / 8 / 1e9
        
        # Adapter memory (if using LoRA)
        if config["rank"]:
            # Approximate LoRA params (attention + FF layers)
            lora_params_per_layer = 4 * d_model * config["rank"] * 2  # Q,K,V,O projections
            total_lora_params = lora_params_per_layer * num_layers
            adapter_memory_gb = total_lora_params * config["adapter_bits"] / 8 / 1e9
        else:
            adapter_memory_gb = 0
            total_lora_params = num_params * config["trainable_ratio"]
        
        total_memory = base_memory_gb + adapter_memory_gb
        trainable_percent = (total_lora_params / num_params) * 100 if config["rank"] else config["trainable_ratio"] * 100
        
        # Suggest GPU
        if total_memory <= 8:
            gpu_needed = "RTX 3070/4060"
        elif total_memory <= 12:
            gpu_needed = "RTX 3080/4070"
        elif total_memory <= 16:
            gpu_needed = "RTX 4080/V100"
        elif total_memory <= 24:
            gpu_needed = "RTX 4090/3090"
        else:
            gpu_needed = "A100/H100"
        
        print(f"{method:<25} {total_memory:>11.1f} {trainable_percent:>11.1f}% {gpu_needed:<15}")

memory_comparison_7b_model()

# ============================================================================
# STEP 8: Simple QLoRA Training Demo
# ============================================================================

print("\n\n🎓 STEP 8: Simple QLoRA Training Demo")
print("-" * 45)

def qlora_training_demo():
    """Demonstrate QLoRA training"""
    
    print("Training a QLoRA model to learn a simple pattern...")
    
    # Create QLoRA model
    qlora_model = QLoRALinear(64, 64, rank=8, alpha=16)
    
    # Only train LoRA parameters
    optimizer = torch.optim.Adam(
        [p for p in qlora_model.parameters() if p.requires_grad], 
        lr=0.01
    )
    criterion = nn.MSELoss()
    
    print(f"Trainable parameters: {sum(p.numel() for p in qlora_model.parameters() if p.requires_grad):,}")
    
    # Training loop - learn to amplify inputs
    target_multiplier = 1.5
    
    for epoch in range(50):
        # Generate training data
        x = torch.randn(16, 64)
        target = x * target_multiplier
        
        # Forward pass
        output = qlora_model(x)
        loss = criterion(output, target)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if epoch % 10 == 0:
            print(f"Epoch {epoch:2d}: Loss = {loss.item():.6f}")
    
    # Test the trained model
    test_x = torch.randn(1, 64)
    with torch.no_grad():
        result = qlora_model(test_x)
        expected = test_x * target_multiplier
        error = torch.abs(result - expected).mean()
    
    print(f"\nTest Results:")
    print(f"Input sample:    {test_x[0, :3].numpy()}")
    print(f"Output sample:   {result[0, :3].numpy()}")  
    print(f"Expected sample: {expected[0, :3].numpy()}")
    print(f"Mean error:      {error:.4f}")
    print("QLoRA training successful! 🎉")

qlora_training_demo()

# ============================================================================
# STEP 9: Key Insights and When to Use QLoRA
# ============================================================================

print("\n\n🎯 STEP 9: Key Insights - When to Use QLoRA")
print("-" * 45)

print("✅ QLoRA is perfect when:")
print("• You have limited GPU memory (4-12 GB)")
print("• You want to fine-tune large models (7B+ parameters)")
print("• You can accept tiny quality loss for huge memory savings")
print("• You want to experiment with large models on consumer hardware")

print("\n❌ Consider alternatives when:")
print("• You have abundant GPU memory (24GB+)")
print("• You need absolute maximum quality")
print("• Training speed is more important than memory")

print("\n🔬 The Science Behind QLoRA Success:")
print("1. Neural networks are robust to quantization noise")
print("2. LoRA adapters compensate for quantization errors")
print("3. 4-bit is the sweet spot (2-bit too lossy, 8-bit unnecessary)")
print("4. NF4 is optimized for neural network weight distributions")

print("\n📈 Typical QLoRA Results:")
print("• Memory: 4-8x reduction vs LoRA")  
print("• Quality: 95-99% of full fine-tuning performance")
print("• Training: Slightly slower than LoRA (quantization overhead)")
print("• Accessibility: Can fine-tune 70B models on single consumer GPU!")

# ============================================================================
# SUMMARY
# ============================================================================

print("\n\n🎯 SUMMARY: QLoRA in Simple Terms")
print("=" * 50)

print("🧠 What is QLoRA?")
print("   LoRA + 4-bit quantization of base model")

print("\n🔧 How it works:")
print("   1. Take a large pre-trained model")
print("   2. Quantize it to 4-bit (NF4 format)")  
print("   3. Freeze the quantized weights")
print("   4. Add small LoRA adapters (16-bit)")
print("   5. Train only the LoRA adapters")

print("\n💾 Memory Magic:")
print("   • Base model: 32-bit → 4-bit (8x reduction)")
print("   • + LoRA adapters: ~1% extra parameters")  
print("   • Total: ~10-20x less memory than full fine-tuning!")

print("\n🎯 The Result:")
print("   • Fine-tune 7B models on 8GB GPUs")
print("   • Fine-tune 70B models on 24GB GPUs")
print("   • Maintain 95-99% of full fine-tuning quality")
print("   • Democratize large model fine-tuning!")

print("\n🚀 QLoRA = LoRA + Smart Compression")
print("   Making AI accessible to everyone! 🌟")